# Lecture 5 (continuous integration): exercise solutions

Worked solutions to the exercises in `continuous-integration.ipynb`. `solution_files/simple_functions/` is the
package at the end of the lecture, with both exercises done: it was produced by following the notes from
`Lecture05/simple_functions/` (`instructor/Lecture05/build/check_ci.py`), and every result below comes from
running its tests and its workflows' commands, in `mpm2026` and in new, empty virtual environments standing in
for GitHub's runners.

## The worked example, for reference

Following the notes: `pytest` reports **7 passed** on your machine. On a fresh runner,
without NumPy in `pyproject.toml`, the test job fails (**2 errors**,
`ModuleNotFoundError: No module named 'numpy'`); with `dependencies = ["numpy"]` it passes
(**7 passed**). If you forget the new line in `__init__.py`, the tests fail on your own machine too:

```text
E   ImportError: cannot import name 'pi' from 'simple_functions'
```

**The pull request "Try it":** after breaking `my_sum` on a branch, the `pytest-unit-tests` check on the pull
request fails (both cases of `test_my_sum`); `flake8` still passes. GitHub warns that checks have failed, but
lets you merge anyway unless a branch protection rule requires them to pass.

## Exercise: `sin(x)`

`simple_functions/trig_functions.py`:

```python
import math

__all__ = ['sin']


def sin(x, terms=20):
    """Return sin(x) for a real number x, from its Taylor series about 0.

    x is first reduced to the interval [-pi, pi] (sin is periodic), where
    `terms` terms of the series are enough for double precision.
    """
    x = math.remainder(x, 2*math.pi)
    total = 0.0
    term = x  # the first term of the series, x**1 / 1!
    for n in range(terms):
        total += term
        term *= -x*x / ((2*n + 2)*(2*n + 3))  # the next term, from this one
    return total
```

plus a line in `simple_functions/__init__.py`:

```python
from .trig_functions import *  # noqa
```

and `tests/test_trig_functions.py`:

```python
import numpy as np
import pytest

from simple_functions import sin


class TestSin:
    """Tests for sin in trig_functions.py."""

    @pytest.mark.parametrize('x', [0.0, 0.5, np.pi/6, np.pi/2, 2.0, np.pi,
                                   4.0, 3*np.pi/2, 6.0, 2*np.pi])
    def test_sin_in_one_period(self, x):
        """sin agrees with numpy for x in [0, 2 pi]."""
        assert sin(x) == pytest.approx(np.sin(x), abs=1e-14)

    @pytest.mark.parametrize('x', [-1.0, -10.0, 10.0, 100.0, 1000.5])
    def test_sin_outside_one_period(self, x):
        """sin agrees with numpy for negative x and x > 2 pi."""
        assert sin(x) == pytest.approx(np.sin(x), abs=1e-12)

    def test_sin_known_values(self):
        """sin(pi/6) = 1/2 and sin(pi/2) = 1."""
        assert sin(np.pi/6) == pytest.approx(0.5)
        assert sin(np.pi/2) == pytest.approx(1.0)
```

`pytest` now reports **23 passed**, locally and on the runner.

**Why the range reduction?** The Taylor series converges for every $x$, but for large $|x|$ its terms grow
enormous before they shrink, and the rounding errors in the big terms swamp the answer. Measured:

```text
max |sin - np.sin| on [0, 2 pi]: 8.0e-16
max |sin - np.sin| on [-1000, 1000]: 3.9e-14
without range reduction, x =  6.283:  4.232461e-16 (np.sin: -2.449294e-16)
without range reduction, x = 10.000: -5.440211e-01 (np.sin: -5.440211e-01)
without range reduction, x = 20.000: -5.364412e+03 (np.sin:  9.129453e-01)
without range reduction, x = 40.000: -7.502111e+15 (np.sin:  7.451132e-01)
complex input, without range reduction: naive(1j) = 1.1752011936438016j, np.sin(1j) = 1.1752011936438014j
sin(1j): TypeError: must be real number, not complex
```

Twenty terms are plenty on $[-\pi, \pi]$, which is all the series ever sees once `math.remainder` (the IEEE
remainder) has reduced $x$ by a multiple of $2\pi$. The small error that remains for large $|x|$ comes from
`2*math.pi` not being exactly $2\pi$. Real libraries reduce the argument far more carefully than this, and use
polynomials tuned for a small interval rather than the Taylor series.

**Tests near zero:** at $x = \pi$, `np.sin(np.pi)` is about $1.2 \times 10^{-16}$, not 0, and our answer differs
from it in the 17th decimal place. A purely relative tolerance fails there (`np.testing.assert_allclose` has
`atol=0` by default), so the tests give an absolute one.

**Complex input:** `math.remainder` only takes real numbers, so `sin(1j)` raises a `TypeError`. Without the
range reduction the series itself works for complex $x$ (the last line above), which is one reason a real
library treats real and complex arguments separately.

## Exercise: a new dependency, and a matrix

A function that needs SciPy, `simple_functions/integration.py`:

```python
from scipy.integrate import quad

__all__ = ['integrate']


def integrate(f, a, b):
    """Return the integral of the function f from a to b.

    A thin wrapper around scipy.integrate.quad, which also returns an
    estimate of the error; here we keep only the value.
    """
    value, _ = quad(f, a, b)
    return value
```

with `from .integration import *  # noqa` in `__init__.py`, and `tests/test_integration.py`:

```python
import numpy as np
import pytest

from simple_functions import integrate


class TestIntegrate:
    """Tests for integrate in integration.py."""

    @pytest.mark.parametrize('f, a, b, expected', [
        (np.sin, 0.0, np.pi, 2.0),
        (lambda x: x**2, 0.0, 3.0, 9.0),
        (np.exp, 0.0, 1.0, np.e - 1.0),
    ])
    def test_integrate(self, f, a, b, expected):
        """integrate gets known integrals right."""
        assert integrate(f, a, b) == pytest.approx(expected)
```

The tests pass on your machine, because `mpm2026` has SciPy. On the runner they fail until SciPy is declared:

```text
E   ModuleNotFoundError: No module named 'scipy'
```

so `pyproject.toml` needs

```toml
dependencies = ["numpy", "scipy"]
```

after which the runner reports **26 passed**. (That is the question "will they pass on the runner?"
in the exercise.)

The matrix, in `.github/workflows/pytest-unit-tests.yml`:

```yml
name: pytest-unit-tests

on:
  # Run the workflow on a push to main, or on a pull request into main
  push:
    branches:
      - main
  pull_request:
    branches:
      - main

jobs:
  pytest:
    name: ${{ matrix.os }}, Python ${{ matrix.python-version }}
    runs-on: ${{ matrix.os }}

    strategy:
      # Let every job in the matrix finish, even if one of them fails
      fail-fast: false
      matrix:
        os: [ubuntu-24.04, ubuntu-26.04, windows-latest, macos-latest]
        python-version: ["3.14"]
        include:
          # One more job: an older Python, on Linux only
          - os: ubuntu-latest
            python-version: "3.12"

    steps:
    - name: Checkout simple_functions
      uses: actions/checkout@v7

    - name: Set up Python ${{ matrix.python-version }}
      uses: actions/setup-python@v7
      with:
        python-version: ${{ matrix.python-version }}

    - name: Install the package and its dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -e ".[dev]"

    - name: Test with pytest
      run: |
        pytest tests/
```

- Five jobs run in parallel: four from the `os` × `python-version` combinations, and one from `include`.
- `fail-fast: false` lets every job finish, so you see every combination that fails, not just the first.
- The job `name` uses the matrix values, so the Actions tab shows which combination each job is.
- `python -m pip install --upgrade pip`, not `pip install --upgrade pip`: on Windows, pip refuses to upgrade
  itself when run as `pip` (the running `pip.exe` cannot be replaced) and asks for `python -m pip`. And a quoted
  `".[dev]"`, which PowerShell, the default shell on Windows runners, accepts as well as `bash`.
- The versions are quoted strings: `"3.14"` and `"3.12"`.
- `ubuntu-26.04` became generally available in September 2026, and `ubuntu-latest` moves to it between 19
  October and 19 November 2026 (until then it is Ubuntu 24.04).

Checked: the workflow passes `actionlint`; the Python 3.12 job's commands pass in a new Python 3.12 environment
(**26 passed**) and the Python 3.14 ones as above. The Windows and macOS jobs can only run on
GitHub.

## Exercise: CI for your Exercise 2 package

`solution_files/mycoolpackage/` now has the same two workflows in `.github/workflows/`. The test workflow runs
`pytest --doctest-modules`, so that the docstring examples are checked too. Run as on a fresh runner:
`flake8` passes (no messages) and the tests pass (**27 passed**).